# Linear Mixed-Effects Model (LMM)
## Fixed Effects of Visual Attention on Perceived Safety

- **N = 1,500** observaciones (30 participantes × 50 imágenes)
- **Variable dependiente:** score de seguridad percibida (1–10)
- **Predictores fijos:** tiempo absoluto de atención (segundos) por clase, estandarizados (z-score)
- **Efecto aleatorio:** intercepto por participante `(1|participante)`
- **Clasificaciones evaluadas:** ADE20K base · Grouped · Disorder · Grouped+Disorder

In [27]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')
import statsmodels.formula.api as smf

print('Librerías OK')

Librerías OK


# ============================================================
# PARÁMETRO: elegir clasificación a analizar
# ============================================================
# Opciones disponibles:
#   'ade20k'    →  ADE20K base          (54 clases originales)
#   'grouped'   →  ADE20K + Grouped     (clases agrupadas semánticamente)
#   'disorder'  →  ADE20K + Disorder    (incluye clases de desorden urbano)
#   'all'       →  Las 4 clasificaciones (más lento, ~5-8 min)
# ============================================================ -->

DATASET = 'ade20k'   # <-- CAMBIA AQUÍ

# Mapa interno (no modificar)
_CONFIGS = {
    'ade20k':   ('main_class',               'ADE20K (base)'),
    'grouped':  ('main_class_grouped',       'ADE20K + Grouped'),
    'disorder': ('main_class_Disorder',      'ADE20K + Disorder'),
    'grouped_disorder': ('main_class_GroupDisorder', 'ADE20K + Grouped + Disorder'),
    'all': None,  # ejecuta los 4
}

if DATASET not in _CONFIGS:
    raise ValueError(f"DATASET='{DATASET}' no válido. Opciones: {list(_CONFIGS.keys())}")

print(f"✓ Dataset seleccionado: {DATASET!r}")
if DATASET != 'all':
    col, lbl = _CONFIGS[DATASET]
    print(f"  columna: {col}")
    print(f"  etiqueta: {lbl}")
else:
    print("  Se ejecutarán las 4 clasificaciones")

## 1. Cargar datos

In [28]:
CSV_PATH    = '../csv/df_final1.csv'
SCORES_PATH = '../json/data_hololens.json'

df = pd.read_csv(CSV_PATH)
with open(SCORES_PATH) as f:
    scores_json = json.load(f)

print(f'Filas CSV:          {len(df):,}')
print(f'Participantes:      {df["participante"].nunique()}')
print(f'Imágenes únicas:    {df["ImageName"].nunique()}')
print(f'Clases ADE20K:      {df["main_class"].nunique()}')

Filas CSV:          869,600
Participantes:      30
Imágenes únicas:    150
Clases ADE20K:      54


## 2. Extraer scores — N = 1,500

In [29]:
score_rows = []
for img_id, img_data in scores_json.items():
    for entry in img_data.get('score_participant', []):
        score_rows.append({
            'ImageName':    int(img_id),
            'participante': int(entry['participant']),
            'score':        float(entry['score'])
        })

scores_df = pd.DataFrame(score_rows)
print(f'N = {len(scores_df)}  (esperado: 1,500)')
print(f'Score  media={scores_df["score"].mean():.3f}  SD={scores_df["score"].std():.3f}  rango=[{int(scores_df["score"].min())}, {int(scores_df["score"].max())}]')
scores_df.head(3)

N = 1500  (esperado: 1,500)
Score  media=5.091  SD=1.997  rango=[1, 10]


,ImageName,participante,score
0,0,2,4.0
1,0,9,4.0
2,0,10,5.0


def run_lmm(class_col, label):
    print(f'\n{"="*72}')
    print(f'  {label}   (columna: {class_col})')
    print(f'{"="*72}')

    # --- 1. Tiempo de atención por clase ---
    df_s = df.sort_values(['participante', 'ImageName', 'Time']).copy()
    df_s['_blk'] = df_s['participante'].astype(str) + '||' + df_s['ImageName'].astype(str)
    df_s['Time_next'] = df_s.groupby('_blk')['Time'].shift(-1)
    df_s['delta_t'] = (df_s['Time_next'] - df_s['Time']).clip(lower=0).fillna(0)
    df_s = df_s[df_s[class_col].notna() & (df_s[class_col].str.strip() != '')].copy()

    att = (
        df_s.groupby(['participante', 'ImageName', class_col])['delta_t']
        .sum().rename('att_sec').reset_index()
        .rename(columns={class_col: 'clase'})
    )

    # --- 2. Pivot + merge scores ---
    pivot = att.pivot_table(
        index=['participante', 'ImageName'],
        columns='clase', values='att_sec',
        aggfunc='first', fill_value=0.0
    ).reset_index()
    pivot.columns.name = None
    model_df = pivot.merge(scores_df, on=['participante', 'ImageName'], how='inner')

    # --- 3. Renombrar + filtrar ---
    class_cols = [c for c in model_df.columns if c not in ['participante', 'ImageName', 'score']]
    rename_map = {c: 'att_' + c.replace(' ', '_').replace('-', '_').replace('/', '_').replace('+', '_')
                  for c in class_cols}
    model_df = model_df.rename(columns=rename_map)
    safe_cols_all = list(rename_map.values())

    min_obs   = len(model_df) * 0.02
    safe_cols = [c for c in safe_cols_all
                 if model_df[c].std() > 1e-6 and (model_df[c] > 0).sum() >= min_obs]
    print(f'  Clases únicas: {len(class_cols)}   Predictores válidos: {len(safe_cols)}')

    # --- 4. Z-score ---
    model_df_std = model_df.copy()
    for c in safe_cols:
        mu, sd = model_df[c].mean(), model_df[c].std()
        model_df_std[c] = (model_df[c] - mu) / sd if sd > 0 else 0.0

    # --- 5. start_params ---
    group_means   = model_df_std.groupby('participante')['score'].mean()
    n_groups      = int(group_means.shape[0])
    grand_mean    = float(model_df_std['score'].mean())
    var_between   = float(np.sum((group_means - grand_mean) ** 2) / (n_groups - 1))
    var_within    = float(model_df_std['score'].var(ddof=1))
    init_var_part = max(var_between - var_within / n_groups, 0.05)

    formula  = 'score ~ ' + ' + '.join(safe_cols)
    ols_coef = smf.ols(formula, data=model_df_std).fit().params.values
    start_params = np.concatenate([ols_coef, [np.sqrt(init_var_part)]])

    # --- 6. Ajustar LMM — estrategia multi-método con criterios de calidad ---
    lmm = smf.mixedlm(formula=formula, data=model_df_std,
                      groups=model_df_std['participante'])

    # Intentos en orden: (método, opciones)
    # nm (Nelder-Mead) va primero porque no usa gradientes → más robusto para modelos pequeños
    attempts = [
        ('nm',     {'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6}),
        ('lbfgs',  {'maxiter': 5000,  'gtol':  1e-5}),
        ('powell', {'maxiter': 5000,  'xtol':  1e-6, 'ftol': 1e-6}),
        ('bfgs',   {'maxiter': 5000,  'gtol':  1e-5}),
        ('cg',     {'maxiter': 5000,  'gtol':  1e-5}),
    ]

    result      = None
    best_result = None      # mejor disponible aunque no cumpla todos los criterios
    best_var    = -1

    for method, opts in attempts:
        try:
            r = lmm.fit(reml=True, method=method, start_params=start_params, options=opts)
            var_p = float(r.cov_re.iloc[0, 0])

            # Estimar calidad del gradiente (solo métodos que lo exponen)
            grad_ok = True
            if hasattr(r, 'mle_retvals') and r.mle_retvals is not None:
                grad = r.mle_retvals.get('grad', None)
                if grad is not None:
                    grad_norm = float(np.linalg.norm(grad))
                    grad_ok   = grad_norm < 1.0   # umbral estricto
                    print(f'  {method}: converged={r.converged}  Part_Var={var_p:.4f}  |grad|={grad_norm:.4f}')
                else:
                    print(f'  {method}: converged={r.converged}  Part_Var={var_p:.4f}')
            else:
                print(f'  {method}: converged={r.converged}  Part_Var={var_p:.4f}')

            # Guardar el mejor disponible
            if var_p > best_var:
                best_var    = var_p
                best_result = r

            # Criterio ideal: convergió + gradiente pequeño + varianza > 0
            if r.converged and grad_ok and var_p > 0.01:
                result = r
                print(f'  → Seleccionado: {method}')
                break
        except Exception as e:
            print(f'  {method}: error — {e}')

    # Fallback: mejor resultado aunque no pase todos los filtros
    if result is None:
        result = best_result
        print(f'  → Fallback: mejor resultado disponible (var_part={best_var:.4f})')

    # --- 7. Resultados ---
    var_part = float(result.cov_re.iloc[0, 0])
    var_res  = float(result.scale)
    icc      = var_part / (var_part + var_res)

    ci  = result.conf_int()
    rev = {v: k for k, v in rename_map.items()}

    rows = []
    for idx in result.params.index:
        if idx == 'Group Var':
            continue
        name = '(Intercept)' if idx == 'Intercept' else rev.get(idx, idx)
        p = result.pvalues[idx]
        rows.append({
            'Predictor': name,
            'Coef.(β)':  round(result.params[idx], 3),
            'S.E.':      round(result.bse[idx], 3),
            'z-val':     round(result.tvalues[idx], 2),
            'p_num':     p,
            'p-val':     '<.001' if p < 0.001 else f'{p:.3f}',
            '95% CI':    f'[{ci.loc[idx,0]:.2f}, {ci.loc[idx,1]:.2f}]'
        })

    table = pd.DataFrame(rows)
    intercept_row = table[table['Predictor'] == '(Intercept)']
    preds = table[table['Predictor'] != '(Intercept)'].copy()
    sig   = preds[preds['p_num'] < 0.05]
    pos   = sig[sig['Coef.(β)'] > 0].sort_values('Coef.(β)', ascending=False)
    neg   = sig[sig['Coef.(β)'] < 0].sort_values('Coef.(β)', ascending=True)
    print(f'  Predictores totales: {len(preds)}   Significativos: {len(sig)} (pos={len(pos)}, neg={len(neg)})')

    H   = f"  {'Predictor':<22} {'Coef.(β)':>9} {'S.E.':>7} {'z-val':>7} {'p-val':>7}  {'95% CI'}"
    SEP = '  ' + '─' * 70

    def fmt(row):
        return (f"  {str(row['Predictor']):<22}"
                f" {row['Coef.(β)']:>9.3f}"
                f" {row['S.E.']:>7.3f}"
                f" {row['z-val']:>7.2f}"
                f" {str(row['p-val']):>7}"
                f"  {row['95% CI']}")

    print(f'\n  Table: Fixed Effects — {label} (N={len(model_df)})')
    print(H); print(SEP)
    for _, r in intercept_row.iterrows():
        print(fmt(r))
    if len(pos) > 0:
        print('\n  Positive (Safe)')
        for _, r in pos.iterrows(): print('  ' + fmt(r))
    if len(neg) > 0:
        print('\n  Negative (Unsafe)')
        for _, r in neg.iterrows(): print('  ' + fmt(r))
    print(SEP)
    print(f"  Random Effects               Var.      SD           ICC")
    print(f"    Participant          {var_part:>9.3f}  {np.sqrt(var_part):>6.3f}        {icc:.3f}")
    print(f"    Residual             {var_res:>9.3f}  {np.sqrt(var_res):>6.3f}")

    return dict(label=label, class_col=class_col,
                n_preds=len(safe_cols), n_sig=len(sig),
                icc=icc, var_part=var_part, var_res=var_res,
                table=table, result=result)

print('Función run_lmm() definida.')

In [30]:
def run_lmm(class_col, label):
    """
    Ajusta LMM para un tipo de clasificación de clase.

    Parameters
    ----------
    class_col : str  — columna de clase en df ('main_class', 'main_class_grouped', ...)
    label     : str  — nombre para mostrar en resultados
    """
    print(f'\n{"="*72}')
    print(f'  {label}   (columna: {class_col})')
    print(f'{"="*72}')

    # --- 1. Tiempo de atención por clase (delta_t acumulado) ---
    df_s = df.sort_values(['participante', 'ImageName', 'Time']).copy()
    df_s['_blk'] = df_s['participante'].astype(str) + '||' + df_s['ImageName'].astype(str)
    df_s['Time_next'] = df_s.groupby('_blk')['Time'].shift(-1)
    df_s['delta_t'] = (df_s['Time_next'] - df_s['Time']).clip(lower=0).fillna(0)
    df_s = df_s[df_s[class_col].notna() & (df_s[class_col].str.strip() != '')].copy()

    att = (
        df_s.groupby(['participante', 'ImageName', class_col])['delta_t']
        .sum().rename('att_sec').reset_index()
        .rename(columns={class_col: 'clase'})
    )

    # --- 2. Pivot → tabla (part × img × clases) + merge scores ---
    pivot = att.pivot_table(
        index=['participante', 'ImageName'],
        columns='clase', values='att_sec',
        aggfunc='first', fill_value=0.0
    ).reset_index()
    pivot.columns.name = None

    model_df = pivot.merge(scores_df, on=['participante', 'ImageName'], how='inner')

    # --- 3. Renombrar columnas para fórmula ---
    class_cols = [c for c in model_df.columns if c not in ['participante', 'ImageName', 'score']]
    rename_map = {c: 'att_' + c.replace(' ', '_').replace('-', '_').replace('/', '_').replace('+', '_')
                  for c in class_cols}
    model_df = model_df.rename(columns=rename_map)
    safe_cols_all = list(rename_map.values())

    # --- 4. Filtrar: al menos 2% de filas con att > 0 y varianza > 0 ---
    min_obs = len(model_df) * 0.02
    safe_cols = [c for c in safe_cols_all
                 if model_df[c].std() > 1e-6 and (model_df[c] > 0).sum() >= min_obs]
    print(f'  Clases únicas: {len(class_cols)}   Predictores válidos: {len(safe_cols)}')

    # --- 5. Estandarizar predictores (z-score) ---
    model_df_std = model_df.copy()
    for c in safe_cols:
        mu, sd = model_df[c].mean(), model_df[c].std()
        model_df_std[c] = (model_df[c] - mu) / sd if sd > 0 else 0.0

    # --- 6. start_params: inicialización alejada del borde var=0 ---
    group_means   = model_df_std.groupby('participante')['score'].mean()
    n_groups      = int(group_means.shape[0])
    grand_mean    = float(model_df_std['score'].mean())
    var_between   = float(np.sum((group_means - grand_mean) ** 2) / (n_groups - 1))
    var_within    = float(model_df_std['score'].var(ddof=1))
    init_var_part = max(var_between - var_within / n_groups, 0.05)

    formula  = 'score ~ ' + ' + '.join(safe_cols)
    ols_coef = smf.ols(formula, data=model_df_std).fit().params.values
    start_params = np.concatenate([ols_coef, [np.sqrt(init_var_part)]])

    # --- 7. Ajustar LMM (REML) ---
    lmm = smf.mixedlm(formula=formula, data=model_df_std,
                      groups=model_df_std['participante'])

    result = None
    for method in ['lbfgs', 'bfgs', 'powell', 'nm']:
        try:
            r = lmm.fit(reml=True, method=method, start_params=start_params)
            var_p = float(r.cov_re.iloc[0, 0])
            if r.converged and var_p > 0.01:
                result = r
                print(f'  Convergió: {method}   Part_Var={var_p:.3f}')
                break
        except Exception:
            pass
    if result is None:  # fallback: aceptar cualquier convergido
        for method in ['lbfgs', 'bfgs', 'powell', 'nm']:
            try:
                r = lmm.fit(reml=True, method=method, start_params=start_params)
                if r.converged:
                    result = r
                    print(f'  Fallback: {method}')
                    break
            except Exception:
                pass

    # --- 8. Extraer resultados ---
    var_part = float(result.cov_re.iloc[0, 0])
    var_res  = float(result.scale)
    icc      = var_part / (var_part + var_res)

    ci  = result.conf_int()
    rev = {v: k for k, v in rename_map.items()}

    rows = []
    for idx in result.params.index:
        if idx == 'Group Var':
            continue
        name = '(Intercept)' if idx == 'Intercept' else rev.get(idx, idx)
        p = result.pvalues[idx]
        rows.append({
            'Predictor': name,
            'Coef.(β)':  round(result.params[idx], 3),
            'S.E.':      round(result.bse[idx], 3),
            'z-val':     round(result.tvalues[idx], 2),
            'p_num':     p,
            'p-val':     '<.001' if p < 0.001 else f'{p:.3f}',
            '95% CI':    f'[{ci.loc[idx,0]:.2f}, {ci.loc[idx,1]:.2f}]'
        })

    table = pd.DataFrame(rows)
    intercept_row = table[table['Predictor'] == '(Intercept)']
    preds = table[table['Predictor'] != '(Intercept)'].copy()
    sig   = preds[preds['p_num'] < 0.05]
    pos   = sig[sig['Coef.(β)'] > 0].sort_values('Coef.(β)', ascending=False)
    neg   = sig[sig['Coef.(β)'] < 0].sort_values('Coef.(β)', ascending=True)
    print(f'  Predictores totales: {len(preds)}   Significativos: {len(sig)} (pos={len(pos)}, neg={len(neg)})')

    # --- 9. Imprimir tabla ---
    H   = f"  {'Predictor':<22} {'Coef.(β)':>9} {'S.E.':>7} {'z-val':>7} {'p-val':>7}  {'95% CI'}"
    SEP = '  ' + '─' * 70

    def fmt(row):
        return (f"  {str(row['Predictor']):<22}"
                f" {row['Coef.(β)']:>9.3f}"
                f" {row['S.E.']:>7.3f}"
                f" {row['z-val']:>7.2f}"
                f" {str(row['p-val']):>7}"
                f"  {row['95% CI']}")

    print(f'\n  Table: Fixed Effects — {label} (N={len(model_df)})')
    print(H); print(SEP)
    for _, r in intercept_row.iterrows():
        print(fmt(r))
    if len(pos) > 0:
        print('\n  Positive (Safe)')
        for _, r in pos.iterrows(): print('  ' + fmt(r))
    if len(neg) > 0:
        print('\n  Negative (Unsafe)')
        for _, r in neg.iterrows(): print('  ' + fmt(r))
    print(SEP)
    print(f"  Random Effects               Var.      SD           ICC")
    print(f"    Participant          {var_part:>9.3f}  {np.sqrt(var_part):>6.3f}        {icc:.3f}")
    print(f"    Residual             {var_res:>9.3f}  {np.sqrt(var_res):>6.3f}")

    return dict(label=label, class_col=class_col,
                n_preds=len(safe_cols), n_sig=len(sig),
                icc=icc, var_part=var_part, var_res=var_res,
                table=table, result=result)

print('Función run_lmm() definida.')

Función run_lmm() definida.


## 4. Ejecutar LMM para las 4 clasificaciones

In [31]:
CLASS_CONFIGS = [
    ('main_class',               'ADE20K (base)'),
    ('main_class_grouped',       'ADE20K + Grouped'),
    ('main_class_Disorder',      'ADE20K + Disorder'),
    ('main_class_GroupDisorder', 'ADE20K + Grouped + Disorder'),
]

lmm_results = {}
for class_col, label in CLASS_CONFIGS:
    res = run_lmm(class_col, label)
    lmm_results[class_col] = res


  ADE20K (base)   (columna: main_class)
  Clases únicas: 54   Predictores válidos: 27
  Convergió: lbfgs   Part_Var=0.665
  Predictores totales: 27   Significativos: 10 (pos=4, neg=6)

  Table: Fixed Effects — ADE20K (base) (N=1500)
  Predictor               Coef.(β)    S.E.   z-val   p-val  95% CI
  ──────────────────────────────────────────────────────────────────────
  (Intercept)                5.091   0.155   32.79   <.001  [4.79, 5.40]

  Positive (Safe)
    tree                       0.195   0.084    2.31   0.021  [0.03, 0.36]
    door                       0.182   0.051    3.57   <.001  [0.08, 0.28]
    palm                       0.130   0.056    2.32   0.020  [0.02, 0.24]
    ashcan                     0.113   0.045    2.50   0.013  [0.02, 0.20]

  Negative (Unsafe)
    wall                      -0.370   0.154   -2.40   0.016  [-0.67, -0.07]
    sky                       -0.188   0.077   -2.43   0.015  [-0.34, -0.04]
    bridge                    -0.159   0.052   -3.06   0.00

if DATASET == 'all':
    configs_to_run = [
        ('main_class',               'ADE20K (base)'),
        ('main_class_grouped',       'ADE20K + Grouped'),
        ('main_class_Disorder',      'ADE20K + Disorder'),
        ('main_class_GroupDisorder', 'ADE20K + Grouped + Disorder'),
    ]
else:
    configs_to_run = [_CONFIGS[DATASET]]

lmm_results = {}
for class_col, label in configs_to_run:
    res = run_lmm(class_col, label)
    lmm_results[class_col] = res

In [32]:
print('='*72)
print('  Resumen comparativo — LMM por tipo de clasificación')
print('='*72)
print(f"  {'Clasificación':<35} {'Pred':>5} {'Sig':>5} {'Part Var':>10} {'Resid Var':>10} {'ICC':>8}")
print('  ' + '─'*68)
for key, r in lmm_results.items():
    print(f"  {r['label']:<35} {r['n_preds']:>5} {r['n_sig']:>5}"
          f" {r['var_part']:>10.3f} {r['var_res']:>10.3f} {r['icc']:>8.3f}")
print('  ' + '─'*68)
print()
print('  Pred = predictores incluidos (≥2% obs, var>0)')
print('  Sig  = predictores significativos (p < .05)')
print('  ICC  = Intraclass Correlation Coefficient (var_part / var_total)')

  Resumen comparativo — LMM por tipo de clasificación
  Clasificación                        Pred   Sig   Part Var  Resid Var      ICC
  ────────────────────────────────────────────────────────────────────
  ADE20K (base)                          27    10      0.665      2.941    0.184
  ADE20K + Grouped                       11     1      0.692      3.180    0.179
  ADE20K + Disorder                      18     6      0.715      2.943    0.196
  ADE20K + Grouped + Disorder            18     6      0.715      2.943    0.196
  ────────────────────────────────────────────────────────────────────

  Pred = predictores incluidos (≥2% obs, var>0)
  Sig  = predictores significativos (p < .05)
  ICC  = Intraclass Correlation Coefficient (var_part / var_total)
